In [26]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

# MACHINE LEARNING MODELS CODE

In [27]:
import os
import cv2
import numpy as np
import pandas as pd

from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

In [28]:
DATASET_PATH = "dataset_v2"

In [29]:
features = []
labels = []

print("Loading Dataset...")

for writer_name in os.listdir(DATASET_PATH):
    writer_path = os.path.join(DATASET_PATH, writer_name)
    if os.path.isdir(writer_path):
        for image_name in os.listdir(writer_path):
            image_path = os.path.join(writer_path, image_name)
            image = cv2.imread(image_path)
            if image is None:
                continue

            image = cv2.resize(image, (128, 128))
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            # HOG FEATURE EXTRACTION
            hog_features = hog(
                gray,
                orientations=9,
                pixels_per_cell=(8, 8),
                cells_per_block=(2, 2),
                block_norm='L2-Hys'
            )

            features.append(hog_features)
            labels.append(writer_name)
features = np.array(features)
labels = np.array(labels)
print("Feature Shape:", features.shape)

Loading Dataset...
Feature Shape: (1162, 8100)


In [30]:

label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    features,
    encoded_labels,
    test_size=0.2,
    random_state=42,
    stratify=encoded_labels
)

In [32]:
ml_models = {

    "SVM": SVC(
        kernel='rbf',
        probability=True
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    "k-NN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "XGBoost": XGBClassifier(
        eval_metric='mlogloss'
    )
}

In [33]:
results = []

for model_name, model in ml_models.items():
    print(f"\nTraining {model_name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(
        y_test,
        y_pred,
        average='weighted',
        zero_division=0.0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average='weighted',
        zero_division=0.0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average='weighted',
        zero_division=0.0
    )

    results.append([
        model_name,
        accuracy,
        precision,
        recall,
        f1
    ])

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)



Training SVM...
Accuracy : 0.49356223175965663
Precision: 0.7290640765002635
Recall   : 0.49356223175965663
F1 Score : 0.5086644191433984

Training Random Forest...
Accuracy : 0.5450643776824035
Precision: 0.5783558708956336
Recall   : 0.5450643776824035
F1 Score : 0.5168136895683276

Training k-NN...
Accuracy : 0.3562231759656652
Precision: 0.5383532724686398
Recall   : 0.3562231759656652
F1 Score : 0.38222666824428736

Training XGBoost...
Accuracy : 0.5150214592274678
Precision: 0.5078993491010659
Recall   : 0.5150214592274678
F1 Score : 0.4853207074014552


In [34]:

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score"
    ]
)

print("\nFinal Results:\n")

print(results_df)


Final Results:

           Model  Accuracy  Precision    Recall  F1-Score
0            SVM  0.493562   0.729064  0.493562  0.508664
1  Random Forest  0.545064   0.578356  0.545064  0.516814
2           k-NN  0.356223   0.538353  0.356223  0.382227
3        XGBoost  0.515021   0.507899  0.515021  0.485321


# CNN MODELS CODE

In [35]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [37]:
IMG_SIZE = 299
BATCH_SIZE = 16
EPOCHS = 10

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cuda


In [38]:
# Image transformations matching standard PyTorch vision models
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load entire dataset
full_dataset = datasets.ImageFolder(
    root=DATASET_PATH,
    transform=train_transform
)

# Deterministic split: 80% train, 20% validation
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True  # Drop last incomplete batch to avoid BatchNorm issues
)

validation_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

NUM_CLASSES = len(full_dataset.class_to_idx)
print("Total Classes:", NUM_CLASSES)

Total Classes: 47


In [44]:
def build_model(base_model):
    # Freeze base model layers
    for param in base_model.parameters():
        param.requires_grad = False
        
    # Replace the classification head dynamically based on network architecture
    if hasattr(base_model, 'fc'): # ResNet, GoogLeNet, InceptionV3
        num_features = base_model.fc.in_features
        base_model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Linear(256, NUM_CLASSES)
        )
    elif hasattr(base_model, 'classifier'): # EfficientNet, ConvNeXt, VGG
        if isinstance(base_model.classifier, nn.Sequential):
            last_idx = len(base_model.classifier) - 1
            if last_idx >= 0 and hasattr(base_model.classifier[last_idx], 'in_features'):
                num_features = base_model.classifier[last_idx].in_features
                base_model.classifier[last_idx] = nn.Sequential(
                    nn.Dropout(0.5),
                    nn.Linear(num_features, 256),
                    nn.ReLU(),
                    nn.Linear(256, NUM_CLASSES)
                )
            else:
                # Fallback replacement for the entire sequential module
                num_features = None
                for layer in base_model.classifier:
                    if isinstance(layer, nn.Linear):
                        num_features = layer.in_features
                        break
                if num_features is None:
                    num_features = 1280
                base_model.classifier = nn.Sequential(
                    nn.Dropout(0.5),
                    nn.Linear(num_features, 256),
                    nn.ReLU(),
                    nn.Linear(256, NUM_CLASSES)
                )
        else:
            num_features = base_model.classifier.in_features
            base_model.classifier = nn.Sequential(
                nn.Dropout(0.5),
                nn.Linear(num_features, 256),
                nn.ReLU(),
                nn.Linear(256, NUM_CLASSES)
            )
    elif hasattr(base_model, 'head'): # Other ConvNeXt versions
        if hasattr(base_model.head, 'fc'):
            num_features = base_model.head.fc.in_features
            base_model.head.fc = nn.Sequential(
                nn.Dropout(0.5),
                nn.Linear(num_features, 256),
                nn.ReLU(),
                nn.Linear(256, NUM_CLASSES)
            )
    
    # Handle auxiliary classifier for InceptionV3 and GoogLeNet
    if hasattr(base_model, 'AuxLogits'):
        if hasattr(base_model.AuxLogits, 'fc'):
            num_aux_features = base_model.AuxLogits.fc.in_features
            base_model.AuxLogits.fc = nn.Linear(num_aux_features, NUM_CLASSES)
        
    return base_model.to(device)

In [45]:
if 'cnn_results' not in globals():
    cnn_results = []

def train_and_evaluate(model_name, model):
    print(f"\nTraining {model_name}...\n")
    criterion = nn.CrossEntropyLoss()
    
    # Only optimize parameters of the custom classifier head (unfrozen ones)
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.0001
    )
    
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            
            # During training, InceptionV3 and GoogLeNet return (logits, aux_logits) tuples
            if isinstance(outputs, tuple):
                # Main output loss
                loss = criterion(outputs[0], labels)
                # Add auxiliary loss with weight 0.3 (standard practice)
                if len(outputs) > 1 and outputs[1] is not None:
                    aux_loss = criterion(outputs[1], labels)
                    loss = loss + 0.3 * aux_loss
                outputs = outputs[0]  # Use main output for accuracy calculation
            else:
                loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        epoch_loss = running_loss / len(train_dataset)
        epoch_acc = correct / total
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {epoch_loss:.4f} - Accuracy: {epoch_acc:.4f}")
        
    # Evaluation
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, labels in validation_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            
            # Under evaluation mode, outputs are always a single Tensor
            if isinstance(outputs, tuple) or not isinstance(outputs, torch.Tensor):
                outputs = outputs[0]
                
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.numpy())
            
    accuracy = accuracy_score(all_targets, all_preds)
    precision = precision_score(all_targets, all_preds, average='weighted', zero_division=0.0)
    recall = recall_score(all_targets, all_preds, average='weighted', zero_division=0.0)
    f1 = f1_score(all_targets, all_preds, average='weighted', zero_division=0.0)
    
    cnn_results.append([
        model_name,
        accuracy,
        precision,
        recall,
        f1
    ])
    
    print(f"\n--- {model_name} Results ---")
    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)
    
    return model

In [43]:
import gc
import warnings

# Suppress auxiliary heads warnings from torchvision
warnings.filterwarnings('ignore', message='.*auxiliary heads.*')
warnings.filterwarnings('ignore', message='.*aux_logits.*')

 # Dictionary containing constructor functions for all 5 pretrained CNN models
cnn_models_to_train = {
    "ResNet50": lambda: models.resnet50(weights=models.ResNet50_Weights.DEFAULT),
    "EfficientNetV2": lambda: models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT),
    "InceptionV3": lambda: models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT, aux_logits=True),
    "GoogLeNet": lambda: models.googlenet(weights=models.GoogLeNet_Weights.DEFAULT, aux_logits=True),
    "ConvNeXt": lambda: models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
}

# Clear any lingering GPU cache before starting the run
gc.collect()
torch.cuda.empty_cache()

# Train and evaluate all 5 models sequentially
for model_name, model_fn in cnn_models_to_train.items():
    print(f"\n==================================================")
    print(f"Starting Training: {model_name}")
    print(f"==================================================")
    
    try:
        # Load and customize model using build_model
        base_model = model_fn()
        model = build_model(base_model)
        
        # Train and evaluate model (showing detailed epochs metrics)
        model = train_and_evaluate(model_name, model)
        
        # Clean up memory immediately to prevent CUDA Out Of Memory
        del base_model, model
    except Exception as e:
        print(f"Error training {model_name}: {e}")
        
    gc.collect()
    torch.cuda.empty_cache()
    
print("\nAll 5 models have been trained and evaluated successfully!")


Starting Training: ResNet50

Training ResNet50...

Epoch 1/10 - Loss: 3.7710 - Accuracy: 0.0764
Epoch 2/10 - Loss: 3.5744 - Accuracy: 0.1023
Epoch 3/10 - Loss: 3.3687 - Accuracy: 0.1367
Epoch 4/10 - Loss: 3.1434 - Accuracy: 0.2476
Epoch 5/10 - Loss: 2.9119 - Accuracy: 0.3638
Epoch 6/10 - Loss: 2.7050 - Accuracy: 0.4435
Epoch 7/10 - Loss: 2.4532 - Accuracy: 0.5630
Epoch 8/10 - Loss: 2.2539 - Accuracy: 0.6319
Epoch 9/10 - Loss: 2.0685 - Accuracy: 0.6997
Epoch 10/10 - Loss: 1.8837 - Accuracy: 0.7395

--- ResNet50 Results ---
Accuracy : 0.7939914163090128
Precision: 0.8434662071357351
Recall   : 0.7939914163090128
F1 Score : 0.7809514933157954

Starting Training: EfficientNetV2

Training EfficientNetV2...

Epoch 1/10 - Loss: 3.7904 - Accuracy: 0.0732
Epoch 2/10 - Loss: 3.6763 - Accuracy: 0.0797
Epoch 3/10 - Loss: 3.5562 - Accuracy: 0.1023
Epoch 4/10 - Loss: 3.4294 - Accuracy: 0.1346
Epoch 5/10 - Loss: 3.2970 - Accuracy: 0.1658
Epoch 6/10 - Loss: 3.1338 - Accuracy: 0.2379
Epoch 7/10 - Loss

In [ ]:
cnn_df = pd.DataFrame(
    cnn_results,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1-Score"]
)
print("\nCNN Models Comparison:\n")
print(cnn_df)


CNN Models Comparison:

            Model  Accuracy  Precision    Recall  F1-Score
0        ResNet50  0.841202   0.885448  0.841202  0.825446
1  EfficientNetV2  0.600858   0.601131  0.600858  0.560625
2        ConvNeXt  0.540773   0.673406  0.540773  0.523582
3       GoogLeNet  0.527897   0.583826  0.527897  0.490623
4        ConvNeXt  0.587983   0.680673  0.587983  0.540420
